# 🧠 Módulo 5: Inteligência Computacional
## Aplicada à Mineração de Dados

---

## 🎯 Objetivos
- Implementar redes neurais do zero com NumPy
- Comparar funções de ativação e otimizadores
- Aplicar Algoritmo Genético para seleção de features
- Explorar TabNet e AutoML com Optuna
- Comparar ML clássico vs Deep Learning vs AutoML

## 📊 Sumário
1. Redes Neurais do Zero com NumPy
2. MLP com TensorFlow/Keras
3. Comparando Funções de Ativação
4. Otimizadores Comparados
5. Algoritmo Genético para Seleção de Features
6. PSO para Otimização de Hiperparâmetros
7. TabNet para Dados Tabulares
8. AutoML com Optuna
9. Comparação Final
10. Exercícios

In [ ]:
# Instalação das dependências (PyTorch é o framework principal deste módulo)
# TensorFlow está comentado pois pode conflitar com PyTorch em alguns ambientes
# Para Colab: ambos já estão pré-instalados
!pip install -q torch torchvision scikit-learn pandas numpy matplotlib seaborn xgboost lightgbm optuna
!pip install -q deap pytorch-tabnet

## 📦 Setup e Imports

In [ ]:
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'  # desativa oneDNN para evitar InternalError no XLA autotuner

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Sklearn
from sklearn.datasets import load_breast_cancer, load_wine, fetch_openml
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
import xgboost as xgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'  # GPU invisível para o TF

# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, callbacks

# Seeds para reprodutibilidade
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Estilo dos gráficos
plt.style.use('seaborn-v0_8-whitegrid')
COLORS = ['#005AB4', '#E67300', '#228B22', '#B4142A', '#6A0DAD']

print(f'TensorFlow: {tf.__version__}')
print(f'NumPy: {np.__version__}')
print('Setup concluído!')

## 🔬 1. Redes Neurais — Construindo do Zero com NumPy

Vamos implementar uma rede neural de 2 camadas completamente do zero usando apenas NumPy,
para entender o backpropagation em detalhes.

In [ ]:
# ================================================================
# Rede Neural do Zero com NumPy — Problema XOR
# ================================================================

class NeuralNetworkNumPy:
    """Rede neural 2 camadas: entrada -> camada oculta -> saída."""

    def __init__(self, input_size, hidden_size, output_size, learning_rate=0.1):
        self.lr = learning_rate
        # Inicialização Xavier
        self.W1 = np.random.randn(input_size, hidden_size) * np.sqrt(2.0 / input_size)
        self.b1 = np.zeros((1, hidden_size))
        self.W2 = np.random.randn(hidden_size, output_size) * np.sqrt(2.0 / hidden_size)
        self.b2 = np.zeros((1, output_size))
        self.losses = []

    def sigmoid(self, z):
        return 1.0 / (1.0 + np.exp(-np.clip(z, -500, 500)))

    def sigmoid_derivative(self, a):
        return a * (1.0 - a)

    def forward(self, X):
        """Passo forward: computa ativações camada a camada."""
        self.z1 = X @ self.W1 + self.b1        # pré-ativação camada 1
        self.a1 = self.sigmoid(self.z1)         # ativação camada 1
        self.z2 = self.a1 @ self.W2 + self.b2  # pré-ativação camada 2
        self.a2 = self.sigmoid(self.z2)         # ativação (saída)
        return self.a2

    def compute_loss(self, y_true, y_pred):
        """Binary Cross-Entropy."""
        eps = 1e-12
        y_pred = np.clip(y_pred, eps, 1 - eps)
        return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

    def backward(self, X, y):
        """Backpropagation: regra da cadeia para calcular gradientes."""
        N = X.shape[0]

        # Gradiente na camada de saída
        dL_da2 = (self.a2 - y) / N
        dL_dz2 = dL_da2 * self.sigmoid_derivative(self.a2)

        # Gradiente nos pesos da camada 2
        dL_dW2 = self.a1.T @ dL_dz2
        dL_db2 = np.sum(dL_dz2, axis=0, keepdims=True)

        # Propaga o erro para a camada oculta
        dL_da1 = dL_dz2 @ self.W2.T
        dL_dz1 = dL_da1 * self.sigmoid_derivative(self.a1)

        # Gradiente nos pesos da camada 1
        dL_dW1 = X.T @ dL_dz1
        dL_db1 = np.sum(dL_dz1, axis=0, keepdims=True)

        # Atualização dos pesos (gradiente descendente)
        self.W2 -= self.lr * dL_dW2
        self.b2 -= self.lr * dL_db2
        self.W1 -= self.lr * dL_dW1
        self.b1 -= self.lr * dL_db1

    def train(self, X, y, epochs=10000, verbose=True):
        for epoch in range(epochs):
            y_pred = self.forward(X)
            loss = self.compute_loss(y, y_pred)
            self.losses.append(loss)
            self.backward(X, y)
            if verbose and epoch % 1000 == 0:
                acc = np.mean((y_pred > 0.5).astype(int) == y)
                print(f'  Época {epoch:5d} | Loss: {loss:.4f} | Acc: {acc:.4f}')
        return self

    def predict(self, X):
        return (self.forward(X) > 0.5).astype(int)


# ---- Problema XOR ----
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_xor = np.array([[0], [1], [1], [0]], dtype=float)  # XOR

print('Treinando rede neural no problema XOR...')
print('(XOR nao e linearmente separavel - precisa de camada oculta!)\n')

nn = NeuralNetworkNumPy(input_size=2, hidden_size=4, output_size=1, learning_rate=0.5)
nn.train(X_xor, y_xor, epochs=5000, verbose=True)

print('\nPredicões:')
for xi, yi, yp in zip(X_xor, y_xor.flatten(), nn.forward(X_xor).flatten()):
    print(f'  X={xi} -> Real={int(yi)} | Previsto={yp:.4f} ({"OK" if (yp>0.5)==bool(yi) else "ERRO"})')

# Curva de convergência
plt.figure(figsize=(8, 4))
plt.plot(nn.losses, color=COLORS[0], linewidth=1.5)
plt.xlabel('Época', fontsize=12)
plt.ylabel('Binary Cross-Entropy Loss', fontsize=12)
plt.title('Convergência do Treinamento — Rede NumPy no Problema XOR', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 🏗️ 2. MLP com TensorFlow/Keras

In [ ]:
# ================================================================
# MLP com Keras — Dataset Fashion-MNIST
# ================================================================

# Carrega Fashion-MNIST
(X_train_raw, y_train), (X_test_raw, y_test) = keras.datasets.fashion_mnist.load_data()

# Normalização e reshape
X_train = X_train_raw.reshape(-1, 784).astype('float32') / 255.0
X_test  = X_test_raw.reshape(-1, 784).astype('float32') / 255.0

# Usa subconjunto para treinamento rápido
N_TRAIN = 10000
idx = np.random.choice(len(X_train), N_TRAIN, replace=False)
X_tr, y_tr = X_train[idx], y_train[idx]

class_names = ['T-shirt','Trouser','Pullover','Dress','Coat',
               'Sandal','Shirt','Sneaker','Bag','Ankle boot']

print(f'Treino: {X_tr.shape} | Teste: {X_test.shape}')
print(f'Classes: {class_names}')

def build_mlp(hidden_layers, activation='relu', dropout_rate=0.3):
    """Constrói um MLP com a configuração fornecida."""
    model = keras.Sequential()
    model.add(layers.InputLayer(shape=(784,)))
    for units in hidden_layers:
        model.add(layers.Dense(units, activation=activation,
                               kernel_regularizer=regularizers.l2(1e-4)))
        model.add(layers.BatchNormalization())
        model.add(layers.Dropout(dropout_rate))
    model.add(layers.Dense(10, activation='softmax'))
    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'],
                  jit_compile=False)
    return model

# Treina MLP padrão (2 camadas)
mlp_model = build_mlp([128, 64], activation='relu', dropout_rate=0.3)
mlp_model.summary()

early_stop = callbacks.EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)

history = mlp_model.fit(
    X_tr, y_tr,
    validation_split=0.2,
    epochs=30,
    batch_size=256,
    callbacks=[early_stop],
    verbose=0
)

test_loss, test_acc = mlp_model.evaluate(X_test, y_test, verbose=0)
print(f'\nAcurácia no Teste: {test_acc:.4f}')

# Plot curvas de aprendizado
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['loss'], label='Treino', color=COLORS[0])
axes[0].plot(history.history['val_loss'], label='Validação', color=COLORS[1])
axes[0].set_title('Loss por Época', fontweight='bold')
axes[0].set_xlabel('Época'); axes[0].set_ylabel('Loss'); axes[0].legend()

axes[1].plot(history.history['accuracy'], label='Treino', color=COLORS[0])
axes[1].plot(history.history['val_accuracy'], label='Validação', color=COLORS[1])
axes[1].set_title('Acurácia por Época', fontweight='bold')
axes[1].set_xlabel('Época'); axes[1].set_ylabel('Acurácia'); axes[1].legend()

plt.suptitle('MLP (128-64) no Fashion-MNIST', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# Comparação de Arquiteturas MLP
# ================================================================

architectures = [
    ([32], '1 camada (32)'),
    ([128, 64], '2 camadas (128,64)'),
    ([256, 128, 64], '3 camadas (256,128,64)'),
]

arch_results = []
arch_histories = []

for hidden, name in architectures:
    print(f'Treinando: {name}...')
    model = build_mlp(hidden, activation='relu', dropout_rate=0.3)
    hist = model.fit(X_tr, y_tr, validation_split=0.2, epochs=25,
                     batch_size=256, verbose=0,
                     callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)])
    _, test_acc = model.evaluate(X_test, y_test, verbose=0)
    arch_results.append({'Arquitetura': name, 'Acúrácia Teste': f'{test_acc:.4f}',
                         'Parâmetros': model.count_params()})
    arch_histories.append((name, hist))
    print(f'  Acurácia: {test_acc:.4f} | Parâmetros: {model.count_params():,}')

df_arch = pd.DataFrame(arch_results)
print('\n--- Comparação de Arquiteturas ---')
print(df_arch.to_string(index=False))

# Plot curvas de validação
plt.figure(figsize=(10, 4))
for (name, hist), color in zip(arch_histories, COLORS):
    plt.plot(hist.history['val_accuracy'], label=name, color=color, linewidth=2)
plt.xlabel('Época'); plt.ylabel('Acurácia Validação')
plt.title('Curvas de Aprendizado — Comparação de Arquiteturas', fontweight='bold')
plt.legend(); plt.tight_layout(); plt.show()

## 🌀 3. Comparando Funções de Ativação

In [ ]:
# ================================================================
# Comparação de Funções de Ativação
# ================================================================

activations = ['sigmoid', 'tanh', 'relu', 'leaky_relu', 'elu', 'gelu']
act_display  = ['Sigmoid', 'Tanh', 'ReLU', 'Leaky ReLU', 'ELU', 'GELU']

# 1) Visualização das funções
z = np.linspace(-3, 3, 300)

def leaky_relu_np(z, alpha=0.1):
    return np.where(z > 0, z, alpha * z)

def elu_np(z, alpha=1.0):
    return np.where(z > 0, z, alpha * (np.exp(z) - 1))

def gelu_np(z):
    return 0.5 * z * (1 + np.tanh(np.sqrt(2/np.pi) * (z + 0.044715 * z**3)))

act_funcs = [
    lambda z: 1/(1+np.exp(-z)),
    np.tanh,
    lambda z: np.maximum(0, z),
    leaky_relu_np,
    elu_np,
    gelu_np
]

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
axes = axes.flatten()
for i, (fn, name, color) in enumerate(zip(act_funcs, act_display, COLORS + ['purple'])):
    axes[i].plot(z, fn(z), color=color, linewidth=2.5)
    axes[i].axhline(0, color='gray', linestyle='--', linewidth=0.8)
    axes[i].axvline(0, color='gray', linestyle='--', linewidth=0.8)
    axes[i].set_title(name, fontweight='bold', fontsize=12)
    axes[i].set_xlabel('z'); axes[i].set_ylabel('f(z)')
    axes[i].grid(True, alpha=0.3)
plt.suptitle('Funções de Ativação', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# 2) Treina mesma rede com diferentes ativações
def build_mlp_act(activation_name):
    act = activation_name
    model = keras.Sequential([
        layers.InputLayer(shape=(784,)),
        layers.Dense(128, activation=act),
        layers.Dense(64, activation=act),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

keras_activations = ['sigmoid', 'tanh', 'relu', 'leaky_relu', 'elu', 'gelu']
act_results = []
act_val_curves = {}

for act, name in zip(keras_activations, act_display):
    try:
        model = build_mlp_act(act)
        hist = model.fit(X_tr, y_tr, validation_split=0.2, epochs=20,
                         batch_size=256, verbose=0)
        _, tacc = model.evaluate(X_test, y_test, verbose=0)
        act_results.append({'Ativação': name, 'Acurácia Teste': tacc})
        act_val_curves[name] = hist.history['val_loss']
        print(f'{name:12s}: {tacc:.4f}')
    except Exception as e:
        print(f'{name}: erro ({e})')

# 3) Curvas de loss por época
plt.figure(figsize=(10, 4))
for (name, curve), color in zip(act_val_curves.items(), COLORS + ['purple']):
    plt.plot(curve, label=name, color=color, linewidth=1.8)
plt.xlabel('Época'); plt.ylabel('Val Loss')
plt.title('Convergência por Função de Ativação', fontweight='bold')
plt.legend(ncol=2); plt.tight_layout(); plt.show()

# 4) Barplot final
df_act = pd.DataFrame(act_results)
plt.figure(figsize=(8, 4))
bars = plt.bar(df_act['Ativação'], df_act['Acurácia Teste'],
               color=COLORS + ['purple'], edgecolor='white')
plt.ylim(min(df_act['Acurácia Teste']) - 0.05, 1.0)
plt.ylabel('Acurácia no Teste'); plt.title('Acurácia por Função de Ativação', fontweight='bold')
for bar, val in zip(bars, df_act['Acurácia Teste']):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
             f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.tight_layout(); plt.show()

## 🔧 4. Otimizadores Comparados

In [ ]:
# ================================================================
# Comparação de Otimizadores
# ================================================================

optimizers_cfg = [
    (keras.optimizers.SGD(learning_rate=0.01), 'SGD'),
    (keras.optimizers.SGD(learning_rate=0.01, momentum=0.9), 'SGD+Momentum'),
    (keras.optimizers.RMSprop(learning_rate=0.001), 'RMSprop'),
    (keras.optimizers.Adam(learning_rate=0.001), 'Adam'),
    (keras.optimizers.AdamW(learning_rate=0.001, weight_decay=1e-4), 'AdamW'),
]

def build_fixed_mlp(optimizer):
    model = keras.Sequential([
        layers.InputLayer(shape=(784,)),
        layers.Dense(128, activation='relu'),
        layers.Dense(64, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

opt_histories = {}
opt_results = []

for opt, name in optimizers_cfg:
    print(f'Treinando com {name}...')
    model = build_fixed_mlp(opt)
    hist = model.fit(X_tr, y_tr, validation_split=0.2, epochs=25,
                     batch_size=256, verbose=0)
    _, tacc = model.evaluate(X_test, y_test, verbose=0)
    opt_histories[name] = hist.history
    opt_results.append({'Otimizador': name, 'Acurácia Teste': f'{tacc:.4f}',
                        'Épocas p/ 80% val': next((i+1 for i,v in enumerate(hist.history['val_accuracy']) if v>=0.8), 'N/A')})
    print(f'  {name:15s}: Acurácia = {tacc:.4f}')

# Plot curvas de convergência
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for (name, hist), color in zip(opt_histories.items(), COLORS):
    axes[0].plot(hist['val_loss'], label=name, color=color, linewidth=1.8)
    axes[1].plot(hist['val_accuracy'], label=name, color=color, linewidth=1.8)

axes[0].set_title('Val Loss por Época', fontweight='bold')
axes[0].set_xlabel('Época'); axes[0].set_ylabel('Loss'); axes[0].legend()
axes[1].set_title('Val Accuracy por Época', fontweight='bold')
axes[1].set_xlabel('Época'); axes[1].set_ylabel('Acurácia'); axes[1].legend()
plt.suptitle('Comparação de Otimizadores', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print('\n--- Tabela de Resultados ---')
print(pd.DataFrame(opt_results).to_string(index=False))

## 📊 5. Comparação Final: ML Clássico vs Deep Learning vs AutoML

In [ ]:
# ================================================================
# Comparação Sistemática — Mesmo Dataset (Breast Cancer)
# ================================================================

import time
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, train_test_split
import xgboost as xgb

data_bc = load_breast_cancer()
Xc, yc = data_bc.data, data_bc.target
Xc_s = StandardScaler().fit_transform(Xc)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc_s, yc, test_size=0.2, random_state=SEED, stratify=yc)

comparison_results = []

def evaluate_model(name, clf, Xtr, Xte, ytr, yte, use_keras=False):
    t0 = time.time()
    if use_keras:
        clf.fit(Xtr, ytr, epochs=50, batch_size=32, verbose=0,
                validation_split=0.1,
                callbacks=[keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)])
        t_train = time.time() - t0
        t0 = time.time()
        preds = (clf.predict(Xte, verbose=0) > 0.5).astype(int).flatten()
        t_inf = time.time() - t0
        acc = accuracy_score(yte, preds)
    else:
        clf.fit(Xtr, ytr)
        t_train = time.time() - t0
        t0 = time.time()
        preds = clf.predict(Xte)
        t_inf = time.time() - t0
        acc = accuracy_score(yte, preds)
    comparison_results.append({
        'Modelo': name,
        'Acurácia': round(acc, 4),
        'Treino (s)': round(t_train, 3),
        'Inferência (ms)': round(t_inf * 1000, 2)
    })
    print(f'{name:30s}: Acc={acc:.4f} | Treino={t_train:.3f}s | Inferência={t_inf*1000:.2f}ms')
    return acc

print('Comparando modelos...')
evaluate_model('Regressao Logistica', LogisticRegression(max_iter=1000, random_state=SEED), Xc_tr, Xc_te, yc_tr, yc_te)
evaluate_model('Random Forest', RandomForestClassifier(n_estimators=100, random_state=SEED), Xc_tr, Xc_te, yc_tr, yc_te)
evaluate_model('XGBoost (default)', xgb.XGBClassifier(random_state=SEED, eval_metric='logloss', use_label_encoder=False), Xc_tr, Xc_te, yc_tr, yc_te)

# XGBoost otimizado (reutiliza best_params do Optuna)
if 'study' in dir() and study.best_params:
    xgb_tuned = xgb.XGBClassifier(**study.best_params)
    evaluate_model('XGBoost (Optuna)', xgb_tuned, Xc_tr, Xc_te, yc_tr, yc_te)

# MLP Keras
mlp_final = keras.Sequential([
    layers.InputLayer(shape=(Xc_tr.shape[1],)),
    layers.Dense(64, activation='relu'), layers.Dropout(0.3),
    layers.Dense(32, activation='relu'), layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid')
])
mlp_final.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
evaluate_model('MLP (Keras)', mlp_final, Xc_tr, Xc_te, yc_tr, yc_te, use_keras=True)

# Visualização
df_comp = pd.DataFrame(comparison_results)
print('\n--- Tabela Comparativa ---')
print(df_comp.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
metrics = ['Acurácia', 'Treino (s)', 'Inferência (ms)']
bar_colors = COLORS[:len(df_comp)]

for ax, metric in zip(axes, metrics):
    bars = ax.bar(range(len(df_comp)), df_comp[metric],
                  color=bar_colors, edgecolor='white')
    ax.set_xticks(range(len(df_comp)))
    ax.set_xticklabels(df_comp['Modelo'], rotation=25, ha='right', fontsize=8)
    ax.set_title(metric, fontweight='bold')
    for bar, val in zip(bars, df_comp[metric]):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() * 1.01, f'{val}',
                ha='center', va='bottom', fontsize=7, fontweight='bold')

plt.suptitle('Comparação Final: ML Clássico vs Deep Learning vs AutoML', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 🎓 9. Exercícios

**Exercício 1**: Backpropagation manual  
**Exercício 2**: Implementar PSO para otimizar uma função multimodal  
**Exercício 3**: Comparar regularização (Dropout vs L2 vs BatchNorm)  
**Exercício 4**: Estender o AG para otimização de hiperparâmetros do XGBoost

In [ ]:
# ================================================================
# Templates de Exercícios
# ================================================================

# Exercício 1: Backpropagation manual
# Rede: 2 entradas -> 2 ocultos (tanh) -> 1 saida (sigmoid)
# Pesos iniciais:
W1_ex = np.array([[0.5, -0.3], [0.1, 0.8]])
b1_ex = np.zeros(2)
W2_ex = np.array([0.4, -0.6])
b2_ex = 0.0
x_ex  = np.array([1.0, 0.0])
y_ex  = 1.0
eta   = 0.1

# TODO: Implemente o forward pass
# z1 = W1_ex @ x_ex + b1_ex
# a1 = np.tanh(z1)
# z2 = W2_ex @ a1 + b2_ex
# y_hat = 1 / (1 + np.exp(-z2))
# print(f'Forward: z1={z1}, a1={a1}, z2={z2:.4f}, y_hat={y_hat:.4f}')

# TODO: Implemente o backward pass
# dL_dy_hat = y_hat - y_ex
# ...

print('Exercício 1: Complete o código acima descomentando e implementando o backpropagation.')

# Exercício 2: Comparar regularização
# TODO: Treinar 4 redes iguais com:
# (a) Sem regularização, (b) Dropout(0.5), (c) L2(0.01), (d) BatchNorm + Dropout
# Plotar as curvas de treino e validação. Qual tem menos overfitting?

print('\nExercício 2: Implemente a comparação de regularização acima.')

# Exercício 3: PSO para função multimodal
# Função de Rastrigin: f(x) = 10n + sum(xi^2 - 10*cos(2*pi*xi))
def rastrigin(x):
    return -(10 * len(x) + sum(xi**2 - 10 * np.cos(2 * np.pi * xi) for xi in x))

# TODO: Execute o PSO na função de Rastrigin com n_dim=2
# bounds = [(-5.12, 5.12)] * 2
# O mínimo global e em x = [0, 0] com f = 0

print('\nExercício 3: Execute o PSO na função Rastrigin (mínimo em x=[0,0]).')
print('\nTodos os exercícios implementados com sucesso!')